# Kling V3 Motion Control API 使用指南

本 Notebook 演示如何通过 **七牛 AIToken** 平台调用 **Kling V3 (Motion Control)** 模型 API。

## 模型简介

Kling V3 Motion Control 支持以下能力：

- **动作控制**：通过参考视频控制生成视频的运动方式
- **图像参考**：通过参考图片控制生成视频的主体外观
- **元素参考**：支持通过 `elements` 传入风格、主体等参考元素
- **音频保留**：支持保留原始视频的音频

**API 端点**：`fal-ai/kling-video/v3/{mode}/motion-control`

**Mode 选项**：`standard`（720P）、`pro`（1080P）

## 1. 环境准备

首先安装 `fal-client` 依赖包，并设置 API Key 和七牛 AIToken 队列服务地址。

In [ ]:
# 安装 fal-client
%pip install fal-client -q

In [ ]:
import os

# 设置七牛 AIToken 队列服务地址
os.environ["FAL_QUEUE_RUN_HOST"] = "api.qnaigc.com/queue"

# 设置 FAL_KEY 环境变量
# 你也可以在终端中通过 export FAL_KEY="xxx" 来设置
os.environ["FAL_KEY"] = os.environ.get("QINIU_API_KEY") or (_ for _ in ()).throw(ValueError("环境变量 QINIU_API_KEY 未设置"))

In [ ]:
import fal_client
from IPython.display import Video, display

## 2. 基础用法 — 队列调用

七牛 AIToken 平台使用队列模式调用 API：先提交请求（`submit`），然后轮询状态（`status`），最后获取结果（`result`）。

下面先封装一个通用的调用辅助函数，后续示例都会复用。

In [ ]:
import time

def run_fal_task(endpoint, arguments, poll_interval=5):
    """提交任务并轮询等待结果返回"""
    # 步骤 1：提交请求
    handler = fal_client.submit(endpoint, arguments=arguments)
    request_id = handler.request_id
    print(f"请求已提交，request_id: {request_id}")

    # 步骤 2：轮询任务状态
    while True:
        status = fal_client.status(endpoint, request_id, with_logs=True)
        print(f"当前状态: {type(status).__name__}")

        if isinstance(status, fal_client.Completed):
            print("任务完成!")
            break

        if hasattr(status, "logs") and status.logs:
            for log in status.logs:
                print(log["message"])

        time.sleep(poll_interval)

    # 步骤 3：获取结果
    return fal_client.result(endpoint, request_id)


# 基础调用示例：参考图片 + 参考视频
result = run_fal_task(
    "fal-ai/kling-video/v3/pro/motion-control",
    arguments={
        "image_url": "https://v3b.fal.media/files/b/0a90ffa7/TNErq9yD7ZxGRATjfAqnh_EIgJSN67.png",
        "video_url": "https://v3b.fal.media/files/b/0a90ff92/hklvF__w53diz6Rve7f5__JuDW2xl0mr6sJ_Kjz3Vxe_vidoeook%20(1)_1.mp4",
        "character_orientation": "image"
    },
)

# 打印返回结果
print(result)

In [ ]:
# 展示生成的视频
video_url = result["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

### 可配置参数一览

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `prompt` | string | 可选 | 描述视频动作的提示词 |
| `image_url` | string | *必填* | 参考图片 URL，用于控制主体外观 |
| `video_url` | string | *必填* | 参考视频 URL，用于控制运动方式 |
| `character_orientation` | string | *必填* | 主体朝向匹配方式：`video`（匹配参考视频，适合复杂动作，最长 30 秒）、`image`（匹配参考图片，适合跟随相机运动，最长 10 秒） |
| `keep_original_sound` | boolean | 可选 | 是否保留原始视频的音频 |
| `elements` | list | 可选 | 元素列表（风格、主体等参考） |

## 3. 高级用法

使用更多可选参数来精细控制视频生成效果。

In [ ]:
# 高级用法：使用更多参数（含 elements）
result_advanced = run_fal_task(
    "fal-ai/kling-video/v3/pro/motion-control",
    arguments={
        "prompt": "A man dancing",
        "image_url": "https://v3b.fal.media/files/b/0a90ffa7/TNErq9yD7ZxGRATjfAqnh_EIgJSN67.png",
        "video_url": "https://v3b.fal.media/files/b/0a90ff92/hklvF__w53diz6Rve7f5__JuDW2xl0mr6sJ_Kjz3Vxe_vidoeook%20(1)_1.mp4",
        "keep_original_sound": True,
        "character_orientation": "image",
        "elements": None
    },
)

# 展示结果
video_url = result_advanced["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 4. 队列模式 — 手动分步调用

上面的 `run_fal_task` 封装了完整的队列流程。如果你需要更精细的控制（例如在提交后做其他事情，稍后再取结果），可以手动分步调用。

In [ ]:
# 步骤 1：提交请求
handler = fal_client.submit(
    "fal-ai/kling-video/v3/pro/motion-control",
    arguments={
        "image_url": "https://v3b.fal.media/files/b/0a90ffa7/TNErq9yD7ZxGRATjfAqnh_EIgJSN67.png",
        "video_url": "https://v3b.fal.media/files/b/0a90ff92/hklvF__w53diz6Rve7f5__JuDW2xl0mr6sJ_Kjz3Vxe_vidoeook%20(1)_1.mp4",
        "character_orientation": "image"
    },
)

# 获取请求 ID，用于后续查询
request_id = handler.request_id
print(f"请求已提交，request_id: {request_id}")

In [ ]:
# 步骤 2：查询任务状态
import time

while True:
    status = fal_client.status(
        "fal-ai/kling-video/v3/pro/motion-control",
        request_id,
        with_logs=True,
    )
    print(f"当前状态: {type(status).__name__}")

    if isinstance(status, fal_client.Completed):
        print("任务完成!")
        break

    # 每 5 种查询一次
    time.sleep(5)

In [ ]:
# 步骤 3：获取结果
result_async = fal_client.result(
    "fal-ai/kling-video/v3/pro/motion-control",
    request_id,
)

# 展示生成的视频
video_url = result_async["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 5. Schema 参考

### Input Schema

```json
{
  "prompt": "string (可选)",
  "image_url": "string (必填, 参考图片 URL)",
  "video_url": "string (必填, 参考视频 URL)",
  "character_orientation": "string (必填, image|video)",
  "keep_original_sound": "boolean (可选)",
  "elements": ["object (可选, 元素列表)"]
}
```

### Output Schema

```json
{
  "video": {
    "file_name": "string",
    "url": "string",
    "content_type": "string",
    "file_size": "number"
  }
}
```

### 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [fal-client Python 文档](https://docs.fal.ai/reference/client-libraries/python/fal_client)
- [Kling V3 模型页面](https://fal.ai/models/fal-ai/kling-video/v3/pro/motion-control)